# ai05 Walkthrough — INSTRUCTOR SOLUTIONS
**DO NOT DISTRIBUTE TO STUDENTS**

## Lesson ai05: APIs & Random Forest
### Sub-Lesson 05a: Fetching Live Data with APIs
### Sub-Lesson 05b: Building Better Models with Random Forest

**Purpose:** Learn to fetch live data from APIs AND build ensemble models that outperform single trees.

**ODE Competencies:** 2.14.1, 2.14.5, 5.1.2

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries imported")

---

# PART 1: APIs — Getting Live Data

## What is an API?

**API = Application Programming Interface**

A contract between your code and a server. You ask for data in a specific format, server sends it back.

Think of it like a restaurant:
- **Menu** = API documentation (what is available?)
- **Your order** = HTTP request (what do you want?)
- **Waiter** = API endpoint (the middleman)
- **Kitchen** = Server/database (where the magic happens)
- **Your food** = JSON response (structured data)

**Key insight:** You do not go into the kitchen yourself. You use the waiter (API) as an intermediary.

### Instructor Note

Students often think APIs magically fetch ALL data. Emphasize:
1. APIs have strict CONTRACTS (parameters matter)
2. You can only ask for what the API supports
3. Bad parameters = empty response or error
4. Always read the docs first!

## URL Structure

```
https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&minmagnitude=5.0&limit=10
Protocol         Host                    Path              Parameters
```

**Parameter rules:**
- `?` starts the parameters
- `key=value` pairs
- `&` separates multiple parameters

In [ ]:
baseUrl = "https://api.open-meteo.com/v1/forecast"

weatherParams = {
    "latitude": 41.14,
    "longitude": -81.86,
    "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
    "temperature_unit": "fahrenheit"
}

weatherResponse = requests.get(baseUrl, params=weatherParams)
print(f"Status code: {weatherResponse.status_code}")

weatherData = weatherResponse.json()
print("✓ Data fetched successfully")

### Instructor Note

Show students what status codes mean:
- 200 = OK (data found)
- 404 = Not found (bad endpoint)
- 403 = Forbidden (need API key)
- 429 = Rate limited (asked too many times)

Ask: "What would happen if you got a 404?"

In [ ]:
dailyData = weatherData["daily"]

weatherDf = pd.DataFrame({
    "date": dailyData["time"],
    "temp_max": dailyData["temperature_2m_max"],
    "temp_min": dailyData["temperature_2m_min"],
    "precipitation": dailyData["precipitation_sum"]
})

weatherDf["date"] = pd.to_datetime(weatherDf["date"])

print("Weather Data for Medina, OH:")
print(weatherDf.head(10))

In [ ]:
eqBaseUrl = "https://earthquake.usgs.gov/fdsnws/event/1/query"

eqParams = {
    "format": "geojson",
    "minmagnitude": 6.0,
    "limit": 10
}

eqResponse = requests.get(eqBaseUrl, params=eqParams)
eqData = eqResponse.json()

earthquakes = []
for feature in eqData["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    earthquakes.append({
        "magnitude": props["mag"],
        "place": props["place"],
        "longitude": coords[0],
        "latitude": coords[1],
        "depth_km": coords[2]
    })

eqDf = pd.DataFrame(earthquakes)
print("Recent Major Earthquakes:")
print(eqDf.to_string())

### Instructor Note

Explain GeoJSON structure:
- "features" = array of objects
- Each object has "properties" (metadata) and "geometry" (location)
- Coordinates follow [longitude, latitude, elevation] order (NOT lat, lng!)

Common mistake: "Why is my latitude negative?" Answer: They swapped coordinates!

---

# PART 2: RANDOM FOREST — ENSEMBLE LEARNING

## Random Forest Overview

**Random Forest** = Collection of decision trees trained on random data + random features.

### Two Layers of Randomness:
1. **Random Samples:** Each tree on different bootstrap sample (~500 rows WITH replacement)
2. **Random Features:** At each split, only consider random subset of features

### Prediction:
- All 100 trees make a prediction
- **Majority vote wins**
- Final prediction = most common answer

### Instructor Note

Key teaching moment: Relate to real life:
- 1 doctor says "You have flu" (single tree = unreliable)
- 100 doctors vote (random forest = consensus)
- If 72 say "flu" and 28 say "cold", the answer is "flu"

Why this works: Each tree overfits in different ways. Averaging reduces bias!

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

nbaData = pd.read_csv("nba_win_prediction.csv")

print(f"Shape: {nbaData.shape}")
print(f"Columns: {list(nbaData.columns)}")
print(f"Baseline accuracy: {(nbaData['home_win'] == 1).mean():.1%}")

In [ ]:
featureColumns = ["pts_diff", "reb_diff", "ast_diff", "fg_pct_diff", "tov_diff", "stl_diff"]
X = nbaData[featureColumns]
y = nbaData["home_win"]

xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {xTrain.shape[0]} games")
print(f"Test set: {xTest.shape[0]} games")

In [ ]:
forestModel = RandomForestClassifier(n_estimators=100, random_state=42)
forestModel.fit(xTrain, yTrain)

yPred = forestModel.predict(xTest)

forestAccuracy = accuracy_score(yTest, yPred)

print(f"\nRandom Forest Accuracy: {forestAccuracy:.2%}")
print(f"Baseline (always home): 54%")
print(f"Improvement: +{(forestAccuracy - 0.54):.2%}")

### Instructor Note

Ask students: "How does the forest know which features matter?"
Answer: Trees vote! If a feature consistently helps trees split well, it gets high importance.

Common misconception: "Feature importance = causation." No! It's just "what helped split."

In [ ]:
featureImportance = pd.DataFrame({
    "Feature": featureColumns,
    "Importance": forestModel.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importance Ranking:")
print(featureImportance.to_string(index=False))

In [ ]:
singleTree = DecisionTreeClassifier(random_state=42)
singleTree.fit(xTrain, yTrain)
yPredSingle = singleTree.predict(xTest)
singleAccuracy = accuracy_score(yTest, yPredSingle)

print(f"\nComparison:")
print(f"Single Tree: {singleAccuracy:.2%}")
print(f"Random Forest (100 trees): {forestAccuracy:.2%}")
print(f"Improvement: +{(forestAccuracy - singleAccuracy):.2%}")

## Summary

### Part 1: APIs
✓ APIs = structured data contracts
✓ URL structure: protocol + host + endpoint + parameters
✓ JSON is nested dicts/lists
✓ requests.get() + .json() = fetch + parse

### Part 2: Random Forest
✓ Ensemble = many experts > one expert
✓ Two randomness layers: data + features
✓ Majority voting combines predictions
✓ Feature importance shows what matters
✓ Better accuracy than single trees